In [ ]:
# @title
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import evaluate
import json
import os

# =========================================================
# KONFIGURATION: Capping anpassen
# =========================================================
DO_CAPPING = False          
MAX_SAMPLES_PER_LABEL = 800  

# =========================================================
# 1. Daten laden
# =========================================================
df = pd.read_csv("Data/English Translated Data/Seniority/seniority_en_full_LLMProf.csv")
df = df.dropna(subset=["text_en", "label"])
df["text_en"] = df["text_en"].astype(str)

print("Original counts:\n", df["label"].value_counts())

# =========================================================
# 1b. Optional: Capping per Label
# =========================================================
if DO_CAPPING:
    dfs = []
    for lbl, cnt in df["label"].value_counts().items():
        take = min(cnt, MAX_SAMPLES_PER_LABEL)
        sub = df[df["label"] == lbl].sample(n=take, random_state=42)
        dfs.append(sub)

    df_limited = pd.concat(dfs, ignore_index=True)
    print("Capped counts:\n", df_limited["label"].value_counts())
else:
    df_limited = df.copy()
    print("No capping: using all samples")

# =========================================================
# 2. Labels encoden
# =========================================================
label_encoder = LabelEncoder()
df_limited["label_id"] = label_encoder.fit_transform(df_limited["label"])
num_labels = df_limited["label_id"].nunique()
print("Labels:", list(label_encoder.classes_))

# =========================================================
# 2b. Class Weights 
# =========================================================
class_counts = df_limited["label_id"].value_counts().sort_index()
alpha = 0.7  
class_weights = 1.0 / np.power(class_counts.values, alpha)
class_weights = class_weights / class_weights.sum()
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)
print("Class counts:", class_counts.to_dict())
print("Class weights:", class_weights_tensor.tolist())

# =========================================================
# 3. Train/Val/Test Split (stratified)
# =========================================================
train_df, temp_df = train_test_split(
    df_limited,
    test_size=0.2,
    stratify=df_limited["label_id"],
    random_state=42,
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label_id"],
    random_state=42,
)

print("Train size:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

# =========================================================
# 4. HuggingFace Datasets
# =========================================================
def to_hf_dataset(df_):
    return Dataset.from_pandas(
        df_[["text_en", "label_id"]].rename(columns={"text_en": "text", "label_id": "label"})
    )

train_dataset = to_hf_dataset(train_df)
val_dataset   = to_hf_dataset(val_df)
test_dataset  = to_hf_dataset(test_df)

# =========================================================
# 5. Tokenizer & Model
# =========================================================
MODEL_NAME = "TechWolf/JobBERT-v3"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset   = val_dataset.map(tokenize_function, batched=True)
test_dataset  = test_dataset.map(tokenize_function, batched=True)

cols = ["input_ids", "attention_mask", "label"]
train_dataset.set_format(type="torch", columns=cols)
val_dataset.set_format(type="torch", columns=cols)
test_dataset.set_format(type="torch", columns=cols)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
)

# =========================================================
# 6. Metrics & TrainingArgs
# =========================================================
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
    }

save_dir = "models/bert_sen_Profs_Jobbert800_uncapped"
os.makedirs(save_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=save_dir,
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=100,
    logging_steps=25,
    eval_steps=250,
    save_steps=250,

    optim="adamw_torch",          
    fp16=False,                    
    dataloader_num_workers=0,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
)
# =========================================================
# 6b. Weighted Trainer
# =========================================================
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(self.model.device)
        self.loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

# =========================================================
# 6c. Trainer initialize
# =========================================================
trainer = WeightedTrainer(
    class_weights=class_weights_tensor,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# =========================================================
# 7. Training + Evaluation
# =========================================================
trainer.train()
metrics_val = trainer.evaluate(val_dataset)
metrics_test = trainer.evaluate(test_dataset)

print("Val-Metriken:", metrics_val)
print("Test-Metriken:", metrics_test)

# =========================================================
# 8. Save
# =========================================================

trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

label_encoder_dict = {
    "classes": list(label_encoder.classes_),
    "mapping": {int(i): cls for i, cls in enumerate(label_encoder.classes_)}
}
with open(f"{save_dir}/label_encoder.json", "w", encoding="utf-8") as f:
    json.dump(label_encoder_dict, f, indent=2, ensure_ascii=False)

print("Done, model saved in:", save_dir)
print("Test Accuracy:", metrics_test["eval_accuracy"])
print("Test F1-macro:", metrics_test["eval_f1_macro"])
print("Capping used:", DO_CAPPING, f"(max {MAX_SAMPLES_PER_LABEL})")

In [ ]:
#@title JobBERT Pro - Uncapped Advanced Training { vertical-output: true }
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import evaluate
import json
import os

# =========================================================
# CONFIGURATION: Advanced Training
# =========================================================
DO_CAPPING = False
MAX_SAMPLES_PER_LABEL = 800
SAVE_NAME = "models/bert_sen_Profs_JobBERT_PRO_uncapped" 

# =========================================================
# 1-5. Daten + Preprocessing 
# =========================================================
df = pd.read_csv("Data/English Translated Data/Seniority/seniority_en_full_LLMProf.csv")
df = df.dropna(subset=["text_en", "label"])
df["text_en"] = df["text_en"].astype(str)

if DO_CAPPING:
    dfs = []
    for lbl, cnt in df["label"].value_counts().items():
        take = min(cnt, MAX_SAMPLES_PER_LABEL)
        sub = df[df["label"] == lbl].sample(n=take, random_state=42)
        dfs.append(sub)
    df_limited = pd.concat(dfs, ignore_index=True)
    print("Capped counts:\n", df_limited["label"].value_counts())
else:
    df_limited = df.copy()
    print("No capping: using all samples")

label_encoder = LabelEncoder()
df_limited["label_id"] = label_encoder.fit_transform(df_limited["label"])
num_labels = df_limited["label_id"].nunique()

class_counts = df_limited["label_id"].value_counts().sort_index()
alpha = 0.7
class_weights = 1.0 / np.power(class_counts.values, alpha)
class_weights = class_weights / class_weights.sum()
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

train_df, temp_df = train_test_split(df_limited, test_size=0.2, stratify=df_limited["label_id"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label_id"], random_state=42)

def to_hf_dataset(df_):
    return Dataset.from_pandas(df_[["text_en", "label_id"]].rename(columns={"text_en": "text", "label_id": "label"}))

train_dataset = to_hf_dataset(train_df)
val_dataset = to_hf_dataset(val_df)
test_dataset = to_hf_dataset(test_df)

MODEL_NAME = "TechWolf/JobBERT-v3"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

cols = ["input_ids", "attention_mask", "label"]
train_dataset.set_format(type="torch", columns=cols)
val_dataset.set_format(type="torch", columns=cols)
test_dataset.set_format(type="torch", columns=cols)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

# =========================================================
# PRO‑METRICS
# =========================================================
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1.compute(predictions=preds, references=labels, average="macro")["f1"],
        "f1_weighted": f1.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

# =========================================================
# PRO‑SAVE DIR
# =========================================================
save_dir = f"/content/drive/MyDrive/PDS_Snapaddy/models/{SAVE_NAME}"
os.makedirs(save_dir, exist_ok=True)
print(f" Saving to: {save_dir}")

# =========================================================
# PRO‑TRAININGARGS 
# =========================================================
training_args = TrainingArguments(
    output_dir=save_dir,

    # Core
    num_train_epochs=4,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,
    gradient_accumulation_steps=2,

    # Learning
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=400,  
    lr_scheduler_type="cosine",

    # Logging
    logging_steps=10,
    eval_steps=100,
    save_steps=200,

    # TPU
    optim="adamw_torch",
    fp16=False,
    dataloader_num_workers=0,
    dataloader_pin_memory=False,

    do_train=True,
    do_eval=True,

    report_to=None,
)

# =========================================================
# WeightedTrainer (unverändert)
# =========================================================
class WeightedTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights.to(self.model.device)
        self.loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

trainer = WeightedTrainer(
    class_weights=class_weights_tensor,
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

# =========================================================
# TRAINING START!
# =========================================================
print(" training started")
trainer.train()

metrics_val = trainer.evaluate(val_dataset)
metrics_test = trainer.evaluate(test_dataset)

print("Val-Metriken:", metrics_val)
print("Test-Metriken:", metrics_test)

# Speichern
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)

label_encoder_dict = {
    "classes": list(label_encoder.classes_),
    "mapping": {int(i): cls for i, cls in enumerate(label_encoder.classes_)}
}
with open(f"{save_dir}/label_encoder.json", "w", encoding="utf-8") as f:
    json.dump(label_encoder_dict, f, indent=2, ensure_ascii=False)

print("pro model saved:", save_dir)
print("Test Accuracy:", metrics_test["eval_accuracy"])
print("Test F1-macro:", metrics_test["eval_f1_macro"])
print("Capping used:", DO_CAPPING)
